## Step 1: Extracting variables from ranalysis (BARRA-R2) and projections (BARPA & CSIRO-CCAM) for specific locations
Detailed description of BARRA paramters here: https://opus.nci.org.au/spaces/NDP/pages/338002591/BARRA2+Parameter+Descriptions 

In [7]:
import warnings
warnings.filterwarnings('ignore')

import xarray as xr
import os
import sys
import dask.distributed
import glob
from dask.distributed import Client
import tempfile
import dask
import numpy as np
import time

# Import utils
sys.path.append('/home/565/dh4185/mn51-dh4185/repos_collab/nesp_bff/')
import utils
# Static metadata dictionaries
from utils import locations, model_dict, cmap_dict, update_locations_orog_sftlf, update_locations, vars_1hr

# Import datafinder
sys.path.append('/home/565/dh4185/mn51-dh4185/repos_collab/dataset_finder/')
from dataset_finder import *

In [2]:
# Dask settings
dask.config.set({
    #'array.chunk-size': "256 MiB",
    #'array.slicing.split_large_chunks': True, 
    'distributed.comm.timeouts.connect': '120s',
    'distributed.comm.timeouts.tcp': '120s',
    'distributed.comm.retry.count': 10,
    'distributed.scheduler.allowed-failures': 20,
    "distributed.scheduler.worker-saturation": 1.1, #< This should use the new behaviour which helps with memory pile up
})

client = Client(n_workers=15, threads_per_worker=1, local_directory = tempfile.mkdtemp(), memory_limit = "32000mb")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/42463/status,
Dashboard: /proxy/42463/status,Workers: 15
Total threads: 15,Total memory: 447.03 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37289,Workers: 0
Dashboard: /proxy/42463/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37775,Total threads: 1
Dashboard: /proxy/41361/status,Memory: 29.80 GiB
Nanny: tcp://127.0.0.1:44087,


In [3]:
# client.close()

In [14]:
##### Settings
# Setting up the metadata for what should be computed
# - Toggle between hourly and daily data
# - Scenarios: historical, ssp126 or ssp370 (Note, BARRA-R2 has only historical data)
# - RCM: BARPA-R, BARRA-R2 or CCAM-v2203-SN
# - Start year: for reference period use 1985, for 2050 use 2035 (Note, BARRA-R2 can't take years post 2022)
# - End year: for reference period use 2014, for 2050 use 2064   (Note, BARRA-R2 can't take years post 2022)
# - Root directory: Computed output is saved here (don't change). Final output directory is depending on the RCM chosen
#####

# Switch between hourly (True) and daily (False) frequency
HOURLY_FREQ = True
_scenario = "ssp370"
_rcm = "BARPA-R"
start_y = 2040
end_y = 2060
interp_30min = False


In [15]:
root_dir = "/g/data/eg3/nesp_bff/step1_raw_data_extraction/"

### List of hourly and daily variables that go in the datafinder check
vars_1hr_list = ['tas','hurs','huss','sfcWind','psl','uas','vas','clt','rsds','rsdsdir','pr']
vars_day_list = ['tasmax','tasmin','huss','psl','sfcWind','sfcWindmax','rsds','rsdsdir','pr']
vars_1hr

{'temperature': ['tas'],
 'humidity_relative': ['hurs'],
 'humidity_specific': ['huss'],
 'wind_speed_10m': ['sfcWind'],
 'pressure': ['psl'],
 'wind_direction_u': ['uas'],
 'wind_direction_v': ['vas'],
 'cloud_cover': ['clt'],
 'solar_direct': ['rsdsdir'],
 'solar_diffuse': ['rsdsdif']}

In [6]:
locations

{'Melbourne': {'Lat': -37.666, 'Lon': 144.832, 'Elev': 118.8},
 'Canberra': {'Lat': -35.305, 'Lon': 149.201, 'Elev': 580.0},
 'Darwin': {'Lat': -12.424, 'Lon': 130.893, 'Elev': 35.0},
 'Cairns': {'Lat': -16.874, 'Lon': 145.746, 'Elev': 8.3},
 'Brisbane': {'Lat': -27.392, 'Lon': 153.129, 'Elev': 9.5},
 'Longreach': {'Lat': -23.437, 'Lon': 144.277, 'Elev': 192.5},
 'Mildura': {'Lat': -34.236, 'Lon': 142.087, 'Elev': 51.1},
 'Adelaide': {'Lat': -34.921, 'Lon': 138.622, 'Elev': 51.0},
 'Perth': {'Lat': -31.927, 'Lon': 115.976, 'Elev': 20.0},
 'Sydney': {'Lat': -33.941, 'Lon': 151.173, 'Elev': 5.0},
 'Hobart': {'Lat': -42.89, 'Lon': 147.328, 'Elev': 51.4}}

In [7]:
utils.update_locations_orog_sftlf(xr.open_dataset(model_dict[_rcm]["sftlf"]).sftlf,
                             xr.open_dataset(model_dict[_rcm]["orog"]).orog,
                             locations,
                             land_fraction_threshold=80,
                             orog_threshold=150,
                             return_gridcell_elev=True)

{'Melbourne': {'Lat': -37.662, 'Lon': 144.8915, 'Elev': 130.3},
 'Canberra': {'Lat': -35.3445, 'Lon': 149.2175, 'Elev': 683.2},
 'Darwin': {'Lat': -12.4785, 'Lon': 130.9865, 'Elev': 17.3},
 'Cairns': {'Lat': -16.959, 'Lon': 145.8185, 'Elev': 147.9},
 'Brisbane': {'Lat': -27.465, 'Lon': 153.08, 'Elev': 32.7},
 'Longreach': {'Lat': -23.448, 'Lon': 144.2735, 'Elev': 192.6},
 'Mildura': {'Lat': -34.263, 'Lon': 142.1105, 'Elev': 50.1},
 'Adelaide': {'Lat': -34.881, 'Lon': 138.557, 'Elev': 63.2},
 'Perth': {'Lat': -31.9455, 'Lon': 116.0, 'Elev': 95.3},
 'Sydney': {'Lat': -33.954, 'Lon': 151.0715, 'Elev': 48.6},
 'Hobart': {'Lat': -42.7605, 'Lon': 147.3635, 'Elev': 175.9},
 'Thredbo': {'Lat': -36.426, 'Lon': 148.2905, 'Elev': 1376.3}}

### Check daily data availability across models
Uses the datafinder tool from ACS to find suitable data. Handy to check if all variables, scenarios and years exist for a given RCM at **daily** timescale. More info here: https://github.com/AusClimateService/dataset_finder 

In [9]:
%%time
#< Specify datasets - do this to find out what models have the required variables
all_data_day = get_datasets("ACS_DS",
                        rcm = _rcm,
                        scenario = ["historical","ssp126","ssp370"],
                        timescale ="day",
                        year = year_range(start_y, end_y))

select_data_day = all_data_day.select(var = vars_day_list, exact_match=True).condense("scenario")
select_data_day

CPU times: user 1min 55s, sys: 16.5 s, total: 2min 11s
Wall time: 2min 24s


,grid,org,gcm,mdl_run,rcm,ver,timescale,,var,date_created,scenario,year,month
0,AUS-15,BOM,ACCESS-CM2,r4i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12
1,AUS-15,BOM,ACCESS-ESM1-5,r6i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12
2,AUS-15,BOM,CESM2,r11i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12
3,AUS-15,BOM,CMCC-ESM2,r1i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12
4,AUS-15,BOM,EC-Earth3,r1i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12
5,AUS-15,BOM,MPI-ESM1-2-HR,r1i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401, v20240801","ssp126, ssp370",2041 to 2060,1 to 12
6,AUS-15,BOM,NorESM2-MM,r1i1p1f1,BARPA-R,v1-r1,day,,"huss, pr, psl, rsds, rsdsdir, sfcWind, sfcWindmax, tasmax, tasmin","latest, v20231001, v20240401","ssp126, ssp370",2041 to 2060,1 to 12


### Check hourly data availability across models
Uses the datafinder tool from ACS to find suitable data. Handy to check if all variables, scenarios and years exist for a given RCM at **hourly** timescale. More info here: https://github.com/AusClimateService/dataset_finder 

In [6]:
%%time
#< Specify datasets - do this to find out what models have the required variables
all_data_1hr = get_datasets("ACS_DS",
                        rcm = _rcm,
                        scenario = ["historical","ssp126","ssp370"],
                        timescale = "1hr",
                        year = year_range(start_y, end_y))

select_data_1hr = all_data_1hr.select(var = vars_1hr_list, exact_match=True).condense("scenario")
select_data_1hr

CPU times: user 42.1 s, sys: 9.3 s, total: 51.4 s
Wall time: 49 s


,grid,org,gcm,mdl_run,rcm,ver,timescale,,var,date_created,scenario,year,month
0,AUS-15,BOM,ACCESS-CM2,r4i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12
1,AUS-15,BOM,ACCESS-ESM1-5,r6i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12
2,AUS-15,BOM,CESM2,r11i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12
3,AUS-15,BOM,CMCC-ESM2,r1i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12
4,AUS-15,BOM,EC-Earth3,r1i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12
5,AUS-15,BOM,MPI-ESM1-2-HR,r1i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401, v20240801","ssp126, ssp370",2061 to 2080,1 to 12
6,AUS-15,BOM,NorESM2-MM,r1i1p1f1,BARPA-R,v1-r1,1hr,,"clt, hurs, huss, pr, psl, rsds, rsdsdir, sfcWind, tas, uas, vas","latest, v20231001, v20240401","ssp126, ssp370",2061 to 2080,1 to 12


In [ ]:
matching_day = select_data_day.find_matches(select_data_1hr, exclude_keys = "timescale")
matching_1hr = select_data_1hr.find_matches(select_data_day, exclude_keys = "timescale")

## Process extraction of variables
Does the same as the executable script __step1_extracting_variables.py__. Good for debugging or calculating individual files

In [17]:
%%time

# Sets timescale, output directory and variable list depending on input in 'Settings' cell at the top
_freq = "1hr" if HOURLY_FREQ else "day"
_vars = vars_1hr if HOURLY_FREQ else vars_day
freqname = "30min" if interp_30min == True else _freq
    
# Output location
if _rcm == "CCAM-v2203-SN":
    out_dir = f"{root_dir}CSIRO-CCAM/"
else:
    out_dir = f"{root_dir}{_rcm}/"

print(f"---------- {_rcm} for '{_freq}' data ----------")
# Corrects location coordinates specified in locations dictionary in utils.py to ensure the selected grid cell from an RCM is on land.  
updated_locations = utils.update_locations_orog_sftlf(xr.open_dataset(model_dict[_rcm]["sftlf"]).sftlf,
                             xr.open_dataset(model_dict[_rcm]["orog"]).orog,
                             locations,
                             land_fraction_threshold=80,
                             orog_threshold=150,
                             return_gridcell_elev=True)

# ======================== MAIN LOOP ============================
# Contains a number of print statements to track progress.

# Iterating though the 12 locations in the updated locations dictionary
for loc in updated_locations:
    start_time_loc = time.time()  # Start timer
    print(f"========================== {loc} =======================")
    lat = updated_locations[loc]['Lat']
    lon = updated_locations[loc]['Lon']
    print(f"Lat: {lat}, Lon: {lon}")
    print(f"---------- {_rcm} for '{_freq}' data ----------")


    gcms_to_process = ["MPI-ESM1-2-HR"]# list(model_dict[_rcm]["gcms"].keys())

    # Iterating through GCMs for the selected RCM
    for _gcm in gcms_to_process:
        start_time_gcm = time.time()
        print(f"***** {_gcm} *****")
        
        # Boolen to specify if data for hourly CCAM has been rechunked
        should_continue = False

        # Specifying output file name in line was naming convention
        out_file = (
            f"{out_dir}{loc}_"
            f"{model_dict[_rcm]['grid']}_"
            f"{_gcm}_{_scenario}_"
            f"{model_dict[_rcm]['gcms'][_gcm]['mdl_run']}_"
            f"{model_dict[_rcm]['org']}_"
            f"{_rcm}_{model_dict[_rcm]['gcms'][_gcm]['version']}_"
            f"{_freq}_{start_y}-{end_y}.nc"
        )

        # Check if file already exists. If a file needs to be recomputed, it has to be deleted manually first
        if not os.path.exists(out_file):
            print(f"Processing: {out_file}.....")
            var_list = []

            # Iterating through the variables (daily or hourly var_list)
            for _var in _vars:
                
                start_time_var = time.time()
                _timescale = _freq
                print(f"{_var}: {_vars[_var]}")

                # Maximum and minimum specific humidity (hussmax, hussmin) is not provided at daily timescale and needs to be 
                # derived from hourly data.
                if _timescale == "day" and _var in ['humidity_specific_max', 'humidity_specific_min']:
                    _timescale = "1hr"
                    
                # BARPA-R is very efficiently chunked four our operation which favours little chunking across time
                # and lots of chunking along lat and lon. CCAM is chunked for each time step but not at all
                # along lat and lon dimensions which requires the dataset to be fully loaded. This takes con-
                # siderable more time to process: BARPA-R day: ~2min, hourly: ~5min. CCAM daily: ~25min, hourly: >7.5h hours
                # Hence, CCAM hourly data is preprocessed to interim files per year, and then loaded and concatenated.                   
                if _rcm == "CCAM-v2203-SN" and _timescale == "1hr" and _var not in ['humidity_specific_max',
                                                                                    'humidity_specific_min']:
                    print(f"Use hourly data for {_var}.")
                    start_time_CCAM_1hr = time.time()
                    # Read proprocessd/rechunked hourly CCAM from /scratch/eg3
                    scratch_dir = f"/scratch/eg3/dh4185/rechunked/{_gcm}/{_scenario}/"
                    rechunk_files = sorted(glob.glob(
                        f"{scratch_dir}{_vars[_var][0]}_"
                        f"{model_dict[_rcm]['grid']}_"
                        f"{_gcm}_{_scenario}_"
                        f"{model_dict[_rcm]['gcms'][_gcm]['mdl_run']}_"
                        f"{model_dict[_rcm]['org']}_{_rcm}_"
                        f"{model_dict[_rcm]['gcms'][_gcm]['version']}_1hr_*.nc"))
                    
                    if len(rechunk_files) != 30 and len(rechunk_files) >= 1:
                        print(f"Files don't cover 30 years from {start_y} to {end_y}. Check files and "
                              f"rerun rechunk_ccam.sh")
                        if rechunk_files:
                            for file in rechunk_files:
                                print(file)
                            should_continue = True
                            break
                    elif len(rechunk_files) == 0:
                        print(f"No files for GCM {_gcm} and {_var} exists. Run "
                              f"rechunk_ccam.sh first.")
                        should_continue = True
                        break
                    else:
                        # print(rechunk_files)
                        # Read all years and preprocessing lat/lon selection
                        da = xr.open_mfdataset(rechunk_files, parallel=True,
                                                            preprocess=lambda ds: utils.preprocess_location(ds, lat, lon))[_vars[_var][0]]
                        da = da.chunk({'time': -1}).sel(time=slice(str(start_y),str(end_y)))
                        # Aliging time coordinates (mix of variables at half hour and full hours)
                        # da_all = utils.process_time(da,_vars[_var][0],_timescale)
                        # da_all = da.resample(time="30min").interpolate("linear")
                        var_list.append(da.to_dataset())
                                                
                        print(f"Processing time for {_var}: {((time.time() - start_time_var)/60):.2f} minutes\n")

                # If BARPA-R or BARRA-R2 at daily or hourly timescale, or CCAM at daily time scale selected process all years at once.
                elif _rcm in ["BARRA-C2","BARPA-R","BARRA-R2"] or _rcm == "CCAM-v2203-SN" and _timescale == "day":
                    if _var == "solar_diffuse":
                        print(f"Use data from /g/data/eg3/nesp_bff/{_rcm}_{_vars[_var]}/ for {_var}.")
                        # Get file paths using the ACS dataset finder
                        all_data = get_datasets("NESP",
                                            rcm=_rcm, gcm=_gcm, scenario=_scenario,
                                            grid=model_dict[_rcm]["grid"],
                                            org=model_dict[_rcm]["org"],
                                            mdl_run=model_dict[_rcm]["gcms"][_gcm]["mdl_run"],
                                            ver=model_dict[_rcm]["gcms"][_gcm]["version"],
                                            timescale=_timescale,
                                            year=year_range(start_y, end_y)).select(var=_vars[_var], exact_match=True)
                    else:
                        print(f"Use data from disk (ia39/kj66) for {_var}.")
                        # Get file paths using the ACS dataset finder
                        all_data = get_datasets("ACS_DS",
                                            rcm=_rcm, gcm=_gcm, scenario=_scenario,
                                            grid=model_dict[_rcm]["grid"],
                                            org=model_dict[_rcm]["org"],
                                            mdl_run=model_dict[_rcm]["gcms"][_gcm]["mdl_run"],
                                            ver=model_dict[_rcm]["gcms"][_gcm]["version"],
                                            timescale=_timescale,
                                            year=year_range(start_y, end_y)).select(var=_vars[_var], exact_match=True)
                    # Read all years and preprocessing lat/lon selection
                    da = xr.open_mfdataset(all_data.get_files(), parallel=True,
                                            preprocess=lambda ds: utils.preprocess_location(ds, lat, lon))[_vars[_var][0]]
                    da = da.chunk({'time': -1})
                    
                    # Using hourly huss data to determine daily hussmax and hussmin
                    if _var == 'humidity_specific_max' or _var == 'humidity_specific_min':
                        da = utils.process_humidity(da,_var)
                    da_all = utils.process_time(da,_vars[_var][0],_timescale) # using 'ceil'
                    # da_all = da.resample(time="30min").interpolate("linear")
                    var_list.append(da_all.to_dataset())
                
                else:
                    print("Inappropriate RCM, GCM, timescale requested. Check Settings.")
                    break
           
                print(f"Processing time for {_var}: {((time.time() - start_time_var)/60):.2f} minutes\n")

            if should_continue:
                print("Move to the next GCM.")
                continue  # Move to the next GCM


            # Remove unwanted variables
            cleaned_list = [da.drop_vars(["bnds", "height", "level_height",
                                          "model_level_number", "sigma"], 
                                          errors="ignore") for da in var_list]

            # Merge data
            da_var = xr.merge(cleaned_list)

            # Compute new rsds as the sum of rsdsdir and rsdsdif
            rsds_new = da_var.rsdsdir + da_var.rsdsdif
            rsds_new.attrs.update({"long_name":"Surface Downwelling Shortwave Radiation",
                                   "standard_name":"surface_downwelling_shortwave_flux_in_air",
                                   "cell_methods":"time: mean (interval: 1 hour)"})
            rsds_new
            da_var_all = da_var.assign(rsds=rsds_new)
            print(da_var_all)

            # saver = da_var_all.to_netcdf(out_file,compute=False)
            # future = client.persist(saver)
            # dask.distributed.progress(future)
            # future.compute()
            # print(f"Saved: {out_file}")
                                
            print(f"Processing time for {_rcm}-{_gcm}: {((time.time() - start_time_gcm)/60):.2f} minutes\n")
            
        else:
            print(f"File already exists: {out_file}")

    print(f"Done with location: {loc}\n")

print("All processing complete.")

---------- BARPA-R for '1hr' data ----------
========================== Melbourne =======================
Lat: -37.662, Lon: 144.8915
---------- BARPA-R for '1hr' data ----------
***** MPI-ESM1-2-HR *****
Processing: /g/data/eg3/nesp_bff/step1_raw_data_extraction/BARPA-R/Melbourne_AUS-15_MPI-ESM1-2-HR_ssp370_r1i1p1f1_BOM_BARPA-R_v1-r1_1hr_2040-2060.nc.....
temperature: ['tas']
Use data from disk (ia39/kj66) for temperature.
INFO: Clash on date_created: Chose "latest" over "v20231001" for var = tas; year = 2041 to 2060; month = 1 to 12


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/xclim/sdba.py:12: UserWarning: The `xclim.sdba` module has been split into its own package `xsdba`. Users are encouraged to use `xsdba` directly. For the time being, `xclim.sdba` will import `xsdba` to allow for API compatibility. This behaviour may change in the future. For more information, see: https://xsdba.readthedocs.io/en/stable/xclim_migration_guide.html
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/xclim/sdba.py:12: UserWarning: The `xclim.sdba` module has been split into its own package `xsdba`. Users are encouraged to use `xsdba` directly. For the time being, `xclim.sdba` will import `xsdba` to allow for API compatibility. This behaviour may change in the future. For more information, see: https://xsdba.readthedocs.io/en/stable/xclim_migration_guide.html
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python

Processing time for temperature: 1.09 minutes

humidity_relative: ['hurs']
Use data from disk (ia39/kj66) for humidity_relative.
INFO: Clash on date_created: Chose "latest" over "v20240401" for var = hurs; year = 2041 to 2060; month = 1 to 12
Processing time for humidity_relative: 0.06 minutes

humidity_specific: ['huss']
Use data from disk (ia39/kj66) for humidity_specific.
INFO: Clash on date_created: Chose "latest" over "v20240401" for var = huss; year = 2041 to 2060; month = 1 to 12
Processing time for humidity_specific: 0.05 minutes

wind_speed_10m: ['sfcWind']
Use data from disk (ia39/kj66) for wind_speed_10m.
INFO: Clash on date_created: Chose "latest" over "v20231001" for var = sfcWind; year = 2041 to 2060; month = 1 to 12
Processing time for wind_speed_10m: 0.06 minutes

pressure: ['psl']
Use data from disk (ia39/kj66) for pressure.
INFO: Clash on date_created: Chose "latest" over "v20231001" for var = psl; year = 2041 to 2060; month = 1 to 12
Processing time for pressure: 0.0

KeyboardInterrupt: 

2026-02-16 15:48:54,159 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/asyncio/base_events.py", line 654, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/core.py", line 494, in finished
    await self._event_finished.wait()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3

In [18]:
t = xr.open_dataset("/g/data/eg3/nesp_bff/step1_raw_data_extraction/BARPA-R/Melbourne_AUS-15_EC-Earth3_ssp370_r1i1p1f1_BOM_BARPA-R_v1-r1_1hr_2060-2080.nc")
t

<xarray.Dataset> Size: 35MB
Dimensions:  (time: 368209)
Coordinates:
  * time     (time) datetime64[ns] 3MB 2060-01-01 ... 2081-01-01
    lat      float64 8B ...
    lon      float64 8B ...
    crs      int32 4B ...
Data variables:
    tas      (time) float64 3MB ...
    hurs     (time) float64 3MB ...
    huss     (time) float64 3MB ...
    sfcWind  (time) float64 3MB ...
    psl      (time) float64 3MB ...
    uas      (time) float64 3MB ...
    vas      (time) float64 3MB ...
    clt      (time) float64 3MB ...
    rsdsdir  (time) float64 3MB ...
    rsdsdif  (time) float64 3MB ...
    rsds     (time) float64 3MB ...